# 数据资产总览

核查 provided 数据、处理后面板以及分析输出是否齐备。


In [ ]:
from pathlib import Path
import json
import sys

import pandas as pd
from IPython.display import Image, Markdown, display

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    if ROOT.name == "notebooks" and (ROOT.parent / "src").exists():
        ROOT = ROOT.parent
    elif (ROOT.parent / "src").exists():
        ROOT = ROOT.parent
    else:
        raise RuntimeError("无法定位仓库根目录，请从项目根目录或 notebooks/ 目录启动 Notebook。")
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

def read_json(path: Path):
    with open(path, "r", encoding="utf-8") as handle:
        return json.load(handle)

def read_api(relpath: str):
    payload = read_json(ROOT / relpath)
    return payload.get("data", payload)

def show_image(relpath: str, width: int = 900):
    display(Image(filename=str(ROOT / relpath), width=width))

def require(*relpaths: str):
    missing = [path for path in relpaths if not (ROOT / path).exists()]
    if missing:
        raise FileNotFoundError(
            "缺少以下产物，请先运行 `PYTHONPATH=src python -m hdi.data.integrator` 和 "
            "`PYTHONPATH=src python -m hdi.analysis.competition`:\n- " + "\n- ".join(missing)
        )


In [ ]:
require(
    "data/raw/provided/01_disease_mortality/全球主要国家死亡原因/2000.csv",
    "data/raw/provided/02_risk_factors/全球各国健康风险因素数据/2000.csv",
    "data/raw/provided/03_nutrition_population/全球各国健康营养和人口统计数据/WB_HNP.csv",
    "data/raw/provided/04_socioeconomic/WDI_CSV/WDICSV.csv",
    "data/raw/provided/05_china_health/全国近20年卫生数据-国家统计局/各省近20年卫生人员数量.csv",
    "data/processed/master_panel.parquet",
    "api_output/metadata/summary.json",
)

datasets = [
    ("疾病死亡数据", "data/raw/provided/01_disease_mortality"),
    ("风险因素数据", "data/raw/provided/02_risk_factors"),
    ("健康营养与人口统计", "data/raw/provided/03_nutrition_population"),
    ("社会经济背景", "data/raw/provided/04_socioeconomic"),
    ("中国卫生数据", "data/raw/provided/05_china_health"),
]

rows = []
for label, relpath in datasets:
    directory = ROOT / relpath
    files = sorted([path for path in directory.rglob("*") if path.is_file() and not path.name.startswith(".")])
    rows.append({"数据集": label, "目录": relpath, "文件数": len(files), "样例文件": files[0].name if files else "无"})

display(pd.DataFrame(rows))


In [ ]:
processed = [
    "data/processed/master_panel.parquet",
    "data/processed/resource_panel.parquet",
    "data/processed/china_panel.parquet",
    "api_output/dim1/trends.json",
    "api_output/dim2/paf.json",
    "api_output/dim3/resource_gap.json",
    "reports/figures/fig01_global_disease_transition.png",
]

status = [{"产物": relpath, "存在": (ROOT / relpath).exists()} for relpath in processed]
display(pd.DataFrame(status))
